# Assignment: Prompt-Based Analysis of a Historical Mini-Corpus

**BSSDH 2026 — Using LLMs in Humanities Research via API**

[![Open in Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LNB-DH/BSSDH_2026_LLM_API_workshop/blob/main/notebooks/assignment_llm_api.ipynb)

In this assignment you will formulate a small humanities research question, write an original prompt, send six historical excerpts to an LLM through OpenRouter, and interpret a results table and two charts.

**Expected time:** 60–90 minutes  
**Required API calls:** normally 6; at most 7 if you revise and retest your prompt  
**Submission:** one executed notebook file

## What you will demonstrate

By completing this notebook, you will show that you can:

- edit and format Markdown cells;
- turn a research idea into explicit inclusion and exclusion criteria;
- supply your own prompt to an API request;
- run prepared Python cells in Google Colab;
- read a Pandas table and Matplotlib chart;
- evaluate LLM-generated evidence critically.

You do **not** need to write functions, loops, Pandas code, JSON parsing code, or plotting code.

## Before you begin

1. Open this notebook in Google Colab.
2. Select **File → Save a copy in Drive**.
3. Rename your copy to **surname_givenname_llm_assignment.ipynb**.
4. Work only in your saved copy.
5. Run the cells from top to bottom.
6. Keep the generated outputs visible when you submit.
7. Never paste your OpenRouter API key into a visible Markdown or code cell.

### Cell guide

- 🟦 **READ** — instructions or explanation
- 🟨 **WRITE** — edit the Markdown cell
- 🟧 **EDIT CODE** — edit only the indicated text
- 🟩 **RUN** — run without editing
- ⭐ **OPTIONAL** — extension for experienced students

If your Colab runtime restarts, variables kept in memory—including your API key and API responses—are lost. Your written Markdown remains saved in the notebook.

## 🟨 Student information

Replace the bracketed text below.

**Name:** [WRITE YOUR NAME]

**Institution or programme:** [WRITE YOUR INSTITUTION OR PROGRAMME]

**Date:** [WRITE THE DATE]

**Notebook filename:** [WRITE THE FINAL FILENAME]

## 🟦 Independent work and responsible use

The six source excerpts are the same for every student. Your independent contribution consists of your:

- research question;
- definition and decision criteria;
- predictions;
- prompt;
- interpretation and critical reflection.

You may discuss the assignment with classmates, but your submitted prompt and written analysis must be your own. Treat the model output as a claim to evaluate, not as an authoritative answer.

### Use of Gemini and other LLM tools

You may use built-in Gemini features in Google Colab or other LLM tools while completing this assignment. This is acceptable as long as you disclose their use. State which tool or model you used, what you used it for, and provide the prompts you entered or a concise summary of those prompts. You remain responsible for checking the suggestions, correcting errors, and ensuring that the submitted research question, prompt choices, and interpretation reflect your own decisions.

Before submitting, complete this disclosure:

**Tool or model used:** [WRITE THE TOOL/MODEL, OR WRITE ?None?]

**What I used it for:** [WRITE A SHORT DESCRIPTION, OR WRITE ?Not applicable?]

**Prompts used or prompt summary:** [PASTE YOUR PROMPTS OR SUMMARIZE THEM, OR WRITE ?Not applicable?]

**How I checked or changed the suggestions:** [WRITE A SHORT DESCRIPTION, OR WRITE ?Not applicable?]

# Part 1 — Set up Colab and load the fixed dataset

Google Colab already includes Pandas and Matplotlib. Run the next cell without installing anything.

<details>
<summary><strong>Running locally instead of Colab?</strong></summary>

This notebook is designed for Colab. For deliberate local use, install the workshop requirements and Matplotlib:

    python -m pip install -r requirements.txt matplotlib

Then open the repository folder so the notebook can find the local dataset.

</details>

In [ ]:
# 🟩 RUN — imports used by the prepared cells
from pathlib import Path
from IPython.display import display, Markdown
import getpass
import hashlib
import io
import json
import re
import time

import requests

try:
    import pandas as pd
    import matplotlib.pyplot as plt
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Pandas or Matplotlib is missing. In Google Colab these are already installed. "
        "For local use, run: python -m pip install pandas matplotlib"
    ) from exc

pd.set_option("display.max_colwidth", 120)
print("Setup imports: PASS")

In [ ]:
# 🟩 RUN — load and verify the fixed six-document mini-corpus
DATA_URL = (
    "https://raw.githubusercontent.com/LNB-DH/"
    "BSSDH_2026_LLM_API_workshop/main/data/assignment_mini_corpus.tsv"
)
EXPECTED_SHA256 = "124988d62ada9b47235d0aea8a70ffc231d9d14b535657b418da10e105c0ff08"
EXPECTED_IDS = [f"LERQ-{number:02d}" for number in range(1, 7)]

local_candidates = [
    Path("data/assignment_mini_corpus.tsv"),
    Path("../data/assignment_mini_corpus.tsv"),
]

data_bytes = None
data_location = None

for candidate in local_candidates:
    if candidate.exists():
        data_bytes = candidate.read_bytes()
        data_location = str(candidate.resolve())
        break

if data_bytes is None:
    response = requests.get(DATA_URL, timeout=30)
    response.raise_for_status()
    data_bytes = response.content
    data_location = DATA_URL

dataset_sha256 = hashlib.sha256(data_bytes).hexdigest()
if dataset_sha256 != EXPECTED_SHA256:
    raise ValueError(
        "Dataset integrity check failed. Reload the original assignment notebook "
        "instead of editing the dataset."
    )

documents = pd.read_csv(io.BytesIO(data_bytes), sep="\t", encoding="utf-8")
actual_ids = documents["document_id"].tolist()

if actual_ids != EXPECTED_IDS:
    raise ValueError(f"Unexpected document IDs: {actual_ids}")
if documents["text"].isna().any() or (documents["text"].str.strip() == "").any():
    raise ValueError("One or more source texts are missing.")
if documents["document_id"].duplicated().any():
    raise ValueError("Duplicate document IDs were found.")

print(f"Dataset location: {data_location}")
print(f"Dataset loaded: {len(documents)} documents")
print("Expected document IDs: PASS")
print("Dataset checksum: PASS")
print(f"Missing texts: {int(documents['text'].isna().sum())}")

In [ ]:
# 🟩 RUN — view the document metadata
display(
    documents[["document_id", "year", "title", "source_filename", "source_uri"]]
    .reset_index(drop=True)
)

## 🟦 About the mini-corpus

These are six curated English-language excerpts from the *Latvian Economic Review* corpus, published between 1936 and 1940. OCR line wrapping has been normalized for readability, and stray photograph-caption lines have been removed. The wording has not been paraphrased. Each row retains its source filename and National Library of Latvia digital-object URI.

Do not edit the dataset or replace its texts. You may inspect, quote, and analyze them.

In [ ]:
# 🟩 RUN — read all six excerpts before designing your prompt
for row in documents.itertuples(index=False):
    display(Markdown(f"### {row.document_id} — {row.title} ({row.year})"))
    display(Markdown(row.text))
    display(Markdown("---"))

# Part 2 — Design your analysis

## 🟨 1. Research question

Replace the bracketed text.

**My research question is:**

[WRITE ONE QUESTION THAT CAN BE INVESTIGATED IN THE SIX EXCERPTS.]

**I chose this question because:**

[WRITE 2–4 SENTENCES.]

**I expect this concept to be relevant to these historical documents because:**

[WRITE 2–4 SENTENCES.]

Possible starting points include economic optimism or anxiety, agricultural modernization, government intervention, international dependence, representations of labour, national identity, and language of crisis or recovery. You may choose a different concept that fits the documents.

## 🟨 2. Operational definition

An operational definition explains how you will decide whether an abstract concept is present in a document.

**Concept being studied:** [WRITE THE CONCEPT]

**Count the concept as present when:**

- [WRITE ONE INCLUSION CRITERION]
- [WRITE A SECOND INCLUSION CRITERION]

**Do not count it as present when:**

- [WRITE ONE EXCLUSION CRITERION]
- [WRITE A SECOND EXCLUSION CRITERION]

**Use “unclear” when:**

[EXPLAIN WHEN THE EVIDENCE WOULD BE TOO AMBIGUOUS.]

## 🟨 3. Make two predictions before calling the model

Return to the displayed excerpts if necessary.

**For document LERQ-01, I predict the label:** [present / absent / unclear]

**My reason:**

[WRITE 1–3 SENTENCES.]

**For document LERQ-06, I predict the label:** [present / absent / unclear]

**My reason:**

[WRITE 1–3 SENTENCES.]

Do not change these predictions after seeing the model output. A disagreement is useful material for analysis.

# Part 3 — Write and test your prompt

Your prompt should explain:

- which concept to identify;
- what counts as evidence;
- what should not count as evidence;
- that the answer must be based only on the supplied historical text.

The response format is added automatically in a later cell. You do not need to write JSON instructions.

In [ ]:
# 🟧 EDIT CODE — replace only the text between the triple quotation marks
my_system_prompt = """
REPLACE THIS TEXT with your own analytical instructions.

Explain your concept, your inclusion criteria, and your exclusion criteria.
Tell the model to base its decision only on the supplied historical text.
"""

In [ ]:
# 🟩 RUN — validate the prompt and append a fixed output contract
prompt_text = my_system_prompt.strip()

if "REPLACE THIS TEXT" in prompt_text:
    raise ValueError("Replace the placeholder with your own prompt before continuing.")
if len(prompt_text) < 180:
    raise ValueError(
        "Your prompt is very short. Add a concept definition plus inclusion and "
        "exclusion criteria; aim for at least 180 characters."
    )

FORMAT_INSTRUCTIONS = """
For each supplied document, return only one valid JSON object with exactly these fields:
{
  "label": "present, absent, or unclear",
  "confidence": 0,
  "evidence": "a short exact quotation from the document, or an empty string",
  "reason": "one concise sentence explaining the decision"
}

The label must be exactly "present", "absent", or "unclear".
Confidence must be a whole number from 0 to 100.
Do not wrap the JSON in Markdown fences. Do not add text before or after the JSON.
"""

final_system_prompt = prompt_text + "\n\n" + FORMAT_INSTRUCTIONS.strip()
current_prompt_id = hashlib.sha256(final_system_prompt.encode("utf-8")).hexdigest()[:12]

print("Prompt customized: PASS")
print(f"Prompt length: {len(prompt_text)} characters")
print(f"Prompt ID: {current_prompt_id}")
print("The fixed JSON output instructions were appended automatically.")

## 🟦 Enter the OpenRouter key safely

The next cell uses a hidden password-style input. The key is stored only in the current runtime memory. It is not printed and is not written into the notebook.

If the runtime restarts, run this cell again. Do not include your key in any written answer or screenshot.

In [ ]:
# 🟩 RUN — enter the key in the hidden input
openrouter_api_key = getpass.getpass("Enter your OpenRouter API key: ").strip()

if not openrouter_api_key:
    raise ValueError("No API key was entered.")
print("OpenRouter API key received: PASS (the key was not displayed)")

In [ ]:
# 🟩 RUN — prepared API and parsing functions; do not edit
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
MODEL = "google/gemini-2.5-flash-lite"
ALLOWED_LABELS = {"present", "absent", "unclear"}

def parse_model_json(raw_text):
    """Parse one model response while retaining failures for inspection."""
    try:
        cleaned = raw_text.strip()
        cleaned = re.sub(r"^\x60{3}(?:json)?\s*", "", cleaned, flags=re.IGNORECASE)
        cleaned = re.sub(r"\s*\x60{3}$", "", cleaned)
        start = cleaned.find("{")
        end = cleaned.rfind("}")
        if start == -1 or end == -1:
            raise ValueError("No JSON object found.")

        parsed = json.loads(cleaned[start:end + 1])
        if not isinstance(parsed, dict):
            raise ValueError("The response is not a JSON object.")

        label = str(parsed.get("label", "")).strip().lower()
        if label not in ALLOWED_LABELS:
            raise ValueError(f"Unexpected label: {label!r}")

        confidence = int(float(parsed.get("confidence")))
        if not 0 <= confidence <= 100:
            raise ValueError("Confidence is outside 0–100.")

        return {
            "label": label,
            "confidence": confidence,
            "evidence": str(parsed.get("evidence", "")).strip(),
            "reason": str(parsed.get("reason", "")).strip(),
            "parse_status": "parsed",
            "parse_error": "",
        }
    except Exception as exc:
        return {
            "label": "",
            "confidence": None,
            "evidence": "",
            "reason": "",
            "parse_status": "failed",
            "parse_error": str(exc),
        }

def call_openrouter(document_id, document_text):
    """Send one document to OpenRouter and retain metadata needed for grading."""
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": final_system_prompt},
            {
                "role": "user",
                "content": (
                    f"Document ID: {document_id}\n\n"
                    f"Historical text:\n{document_text}"
                ),
            },
        ],
        "temperature": 0.1,
        "max_tokens": 350,
    }
    headers = {
        "Authorization": f"Bearer {openrouter_api_key}",
        "Content-Type": "application/json",
        "HTTP-Referer": "https://www.digitalhumanities.lv/bssdh/2026/",
        "X-Title": "BSSDH 2026 LLM API Assignment",
    }

    try:
        response = requests.post(
            OPENROUTER_URL,
            headers=headers,
            json=payload,
            timeout=90,
        )
        response.raise_for_status()
        api_result = response.json()
        raw_text = api_result["choices"][0]["message"]["content"]
        return {
            "document_id": document_id,
            "raw_response": raw_text,
            "request_status": "received",
            "request_error": "",
            "model": api_result.get("model", MODEL),
            "prompt_id": current_prompt_id,
            "usage": api_result.get("usage", {}),
        }
    except Exception as exc:
        return {
            "document_id": document_id,
            "raw_response": "",
            "request_status": "failed",
            "request_error": str(exc),
            "model": MODEL,
            "prompt_id": current_prompt_id,
            "usage": {},
        }

if "raw_responses" not in globals():
    raw_responses = {}

print(f"API helper ready. Core assignment model: {MODEL}")

## 🟦 Test the prompt on one document

The next cell makes **one API call** only after you type **RUN**.

Inspect the returned label, evidence, and reason. If the response does not match your intended criteria, revise **my_system_prompt**, rerun the prompt-validation cell, and rerun this test. The new prompt ID lets the notebook detect an older result.

In [ ]:
# 🟩 RUN — test on LERQ-01; type RUN only when ready
test_row = documents.loc[documents["document_id"] == "LERQ-01"].iloc[0]
existing_test = raw_responses.get(test_row["document_id"])
same_prompt = (
    existing_test is not None
    and existing_test.get("prompt_id") == current_prompt_id
    and existing_test.get("request_status") == "received"
)

if same_prompt:
    print("A successful test response for the current prompt already exists.")
    action = input("Press Enter to keep it, or type RUN to replace it with a new API call: ")
else:
    action = input("Type RUN to make one test API call, or press Enter to cancel: ")

if action.strip().upper() == "RUN":
    print(f"Calling {MODEL} for {test_row['document_id']}...")
    raw_responses[test_row["document_id"]] = call_openrouter(
        test_row["document_id"],
        test_row["text"],
    )
else:
    print("No API call was made.")

test_record = raw_responses.get(test_row["document_id"])
if test_record and test_record.get("prompt_id") == current_prompt_id:
    if test_record["request_status"] == "received":
        test_parsed = parse_model_json(test_record["raw_response"])
        display(pd.DataFrame([{**{"document_id": test_row["document_id"]}, **test_parsed}]))
    else:
        print("The API request failed:", test_record["request_error"])

## 🟨 4. Review the test response

Complete this after making the test call.

**The test label was:** [WRITE THE LABEL]

**The evidence was or was not copied accurately from the source because:**

[CHECK THE SOURCE TEXT AND WRITE 1–3 SENTENCES.]

**The result followed my intended criteria:** [yes / partly / no]

**I revised my prompt after this test:** [yes / no]

**If yes, what did I change and why?**

[WRITE YOUR ANSWER, OR WRITE “Not applicable.”]

If you revised the prompt, rerun the orange prompt cell, the validation cell, and the test cell before continuing.

# Part 4 — Analyze all six documents

The next cell identifies which documents still need a response for the current prompt. It will reuse the successful test result and normally make five more calls.

It will not silently spend API credit: you must type **RUN ALL** before it sends requests. Rerunning the cell will reuse successful responses with the same prompt ID.

In [ ]:
# 🟩 RUN — complete the six-document analysis
successful_current_ids = {
    document_id
    for document_id, record in raw_responses.items()
    if record.get("prompt_id") == current_prompt_id
    and record.get("request_status") == "received"
}
pending_rows = documents[
    ~documents["document_id"].isin(successful_current_ids)
]

if pending_rows.empty:
    print("All six successful responses already exist for the current prompt.")
else:
    print(
        f"{len(pending_rows)} API call(s) are needed for prompt ID "
        f"{current_prompt_id}: {pending_rows['document_id'].tolist()}"
    )
    confirmation = input("Type RUN ALL to continue, or press Enter to cancel: ")

    if confirmation.strip().upper() == "RUN ALL":
        for position, row in enumerate(pending_rows.itertuples(index=False), start=1):
            print(f"[{position}/{len(pending_rows)}] Calling the model for {row.document_id}...")
            record = call_openrouter(row.document_id, row.text)
            raw_responses[row.document_id] = record
            print(f"    {record['request_status']}")
            if record["request_status"] == "failed":
                print(f"    Error: {record['request_error']}")
            time.sleep(0.5)
        print("Batch finished.")
    else:
        print("No batch API calls were made.")

In [ ]:
# 🟩 RUN — convert the current responses into a grading-friendly table
result_rows = []

for source_row in documents.itertuples(index=False):
    record = raw_responses.get(source_row.document_id)

    if record is None or record.get("prompt_id") != current_prompt_id:
        result_rows.append({
            "document_id": source_row.document_id,
            "year": source_row.year,
            "title": source_row.title,
            "label": "",
            "confidence": None,
            "evidence": "",
            "reason": "",
            "request_status": "missing",
            "parse_status": "not attempted",
            "parse_error": "",
            "total_tokens": None,
        })
        continue

    if record.get("request_status") != "received":
        result_rows.append({
            "document_id": source_row.document_id,
            "year": source_row.year,
            "title": source_row.title,
            "label": "",
            "confidence": None,
            "evidence": "",
            "reason": "",
            "request_status": "failed",
            "parse_status": "not attempted",
            "parse_error": record.get("request_error", ""),
            "total_tokens": None,
        })
        continue

    parsed = parse_model_json(record["raw_response"])
    result_rows.append({
        "document_id": source_row.document_id,
        "year": source_row.year,
        "title": source_row.title,
        **parsed,
        "request_status": "received",
        "total_tokens": record.get("usage", {}).get("total_tokens"),
    })

results = pd.DataFrame(result_rows)
print("Results table created.")

In [ ]:
# 🟩 RUN — inspect the table carefully
display_columns = [
    "document_id",
    "year",
    "title",
    "label",
    "confidence",
    "evidence",
    "reason",
    "parse_status",
]
display(results[display_columns].reset_index(drop=True))

failed_parses = results[results["parse_status"] == "failed"]
if not failed_parses.empty:
    print("\nResponses that could not be parsed automatically:")
    for document_id in failed_parses["document_id"]:
        print(f"\n--- {document_id} raw response ---")
        print(raw_responses[document_id]["raw_response"])

## 🟦 Read the charts

The first chart counts the three labels. The second displays the model's own confidence number for each document.

**Important:** model-reported confidence is not a measured or calibrated probability. A high number does not prove that an answer is correct. Check the quoted evidence against the original document.

In [ ]:
# 🟩 RUN — generate two prepared Matplotlib charts
label_order = ["present", "absent", "unclear"]
label_counts = (
    results.loc[results["parse_status"] == "parsed", "label"]
    .value_counts()
    .reindex(label_order, fill_value=0)
)

parsed_results = results[results["parse_status"] == "parsed"].copy()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

colors = ["#2a9d8f", "#e76f51", "#e9c46a"]
label_counts.plot(kind="bar", ax=axes[0], color=colors)
axes[0].set_title("Model labels across the mini-corpus")
axes[0].set_xlabel("Label")
axes[0].set_ylabel("Number of documents")
axes[0].tick_params(axis="x", rotation=0)
axes[0].set_ylim(0, max(6, int(label_counts.max()) + 1))

if not parsed_results.empty:
    axes[1].bar(
        parsed_results["document_id"],
        parsed_results["confidence"],
        color="#457b9d",
    )
axes[1].set_title("Model-reported confidence by document")
axes[1].set_xlabel("Document ID")
axes[1].set_ylabel("Self-reported confidence (0–100)")
axes[1].set_ylim(0, 100)
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

# Part 5 — Interpret the results

## 🟨 5. Results analysis

Answer all questions. Refer to exact counts and document IDs.

### A. Overall pattern

**The most common label was [LABEL], applied to [NUMBER] documents.**

[WRITE 2–4 SENTENCES EXPLAINING THE OVERALL PATTERN.]

### B. One convincing result

**Document ID:** [WRITE ID]

[EXPLAIN WHY THE LABEL IS CONVINCING. CHECK THE MODEL'S QUOTED EVIDENCE AGAINST THE SOURCE.]

### C. One questionable or ambiguous result

**Document ID:** [WRITE ID]

[EXPLAIN WHAT IS QUESTIONABLE, MISSING, OR OPEN TO ANOTHER INTERPRETATION.]

### D. Compare the model with your predictions

[DISCUSS LERQ-01 AND/OR LERQ-06. A DISAGREEMENT IS NOT A MISTAKE; EXPLAIN IT.]

### E. Prompt influence

[EXPLAIN ONE WAY YOUR PROMPT WORDING OR CRITERIA MAY HAVE SHAPED THE RESULTS.]

## 🟨 6. Critical reflection and limitations

**What can we not conclude from only six excerpts?**

[WRITE 2–4 SENTENCES.]

**Identify one limitation related to the historical sources, OCR, prompt, model, or classification method.**

[WRITE 2–4 SENTENCES.]

**What would you change before applying this method to a full corpus?**

[WRITE 2–4 SENTENCES.]

**In this assignment, what was more useful: the model's label, quoted evidence, reason, or something else? Why?**

[WRITE 2–4 SENTENCES.]

# ⭐ Optional extensions

The core assignment is complete without these. If you have additional experience or curiosity, you may:

- revise the prompt and compare which labels change;
- compare two models on two documents;
- manually evaluate all six quoted pieces of evidence;
- change the chart design;
- investigate whether the model treats statistical and rhetorical evidence differently;
- continue with the optional [`workshop_session_3.ipynb`](workshop_session_3.ipynb) for historical OCR, translation, and image-input experiments.

Clearly label any optional work so it is not confused with the required analysis.

# Part 6 — Submission

## 🟨 Final checklist

Before downloading your notebook, confirm each item by replacing **[ ]** with **[x]**.

- [ ] I saved the notebook under the required filename.
- [ ] I completed the student information.
- [ ] I wrote my own research question and operational definition.
- [ ] I made predictions before interpreting the model output.
- [ ] I wrote my own prompt.
- [ ] The dataset integrity check passed.
- [ ] At least five API responses were received.
- [ ] I ran the table, chart, and final summary cells.
- [ ] I referred to counts and document IDs in my analysis.
- [ ] I completed the critical reflection.
- [ ] My API key is not visible anywhere in the notebook.
- [ ] The notebook outputs are visible.

Then select **File → Download → Download .ipynb** in Colab and submit that single file.

In [ ]:
# 🟩 RUN LAST — compact technical summary for you and the grader
dataset_ok = (
    dataset_sha256 == EXPECTED_SHA256
    and documents["document_id"].tolist() == EXPECTED_IDS
    and len(documents) == 6
)
prompt_ok = "REPLACE THIS TEXT" not in prompt_text and len(prompt_text) >= 180
received_count = int((results["request_status"] == "received").sum())
parsed_count = int((results["parse_status"] == "parsed").sum())
label_summary = (
    results.loc[results["parse_status"] == "parsed", "label"]
    .value_counts()
    .reindex(["present", "absent", "unclear"], fill_value=0)
    .to_dict()
)

models_used = sorted({
    record.get("model", MODEL)
    for record in raw_responses.values()
    if record.get("prompt_id") == current_prompt_id
    and record.get("request_status") == "received"
})

print("=" * 58)
print("ASSIGNMENT TECHNICAL SUMMARY")
print("=" * 58)
print(f"Dataset integrity: {'PASS' if dataset_ok else 'FAIL'}")
print(f"Documents in fixed dataset: {len(documents)}")
print(f"Prompt customized: {'PASS' if prompt_ok else 'FAIL'}")
print(f"Prompt ID: {current_prompt_id}")
print(f"Prompt length: {len(prompt_text)} characters")
print(f"Responses received: {received_count}/6")
print(f"Responses parsed: {parsed_count}/6")
print(
    "Labels: "
    + ", ".join(f"{label}={count}" for label, count in label_summary.items())
)
print(f"Model response IDs: {models_used or [MODEL]}")
print("API key handling: hidden runtime input used; inspect visible cells before submission")
print("Required Markdown sections: CHECK MANUALLY")
print("=" * 58)

if not dataset_ok:
    print("ACTION NEEDED: reload the original dataset/notebook.")
if not prompt_ok:
    print("ACTION NEEDED: customize and revalidate the prompt.")
if received_count < 5:
    print("ACTION NEEDED: at least five responses are required.")
if parsed_count < received_count:
    print("NOTE: inspect any raw response that could not be parsed.")